# Manual ADP Update

One-shot notebook for the manual-upload workflow (see `docs/manual-upload-playbook.md`).

**How to use:**
1. Drop your three rankings exports (DK, UD, Drafters) somewhere on your machine, e.g. the repo root.
2. Edit the CONFIG cell below with today's date and the three filenames.
3. Run All. The notebook syncs git, sanity-checks the DK file is NFL, strips today's stale auto rows, appends manual rows, rebuilds `latest.json`, and commits/pushes.
4. If any cell fails, fix the issue and re-run from that cell down.

FFPC stays as stale auto — no FFPC manual file expected.

## 1. CONFIG — edit this cell

In [ ]:
# Today's date (YYYY-MM-DD).
TODAY = "2026-07-18"

# Where the three drop files currently live. Default is the repo root.
# If you dropped them elsewhere, change DROPS_DIR.
DROPS_DIR = ".."  # relative to scripts/ where this notebook lives

# Filenames of today's drops. Just the basename, not the full path.
UD_FILE       = "rankings-a9c04e81-1ace-4b16-a31d-4c725a47f16f-ccf300b0-9197-5951-bd96-cba84ad71e86(20).csv"
DK_FILE       = "DkPreDraftRankings(50).csv"
DRAFTERS_FILE = "drafters_players(18).csv"

## 2. Setup — imports, constants, paths

In [ ]:
import csv, json, shutil, subprocess, sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

# Repo root = parent of scripts/ where this notebook lives.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "scripts":
    REPO_ROOT = REPO_ROOT.parent

DASHBOARD_DIR    = REPO_ROOT / "dashboards" / "best-ball-prices"
DK_HISTORY       = DASHBOARD_DIR / "dk_adp_history.csv"
UD_HISTORY       = DASHBOARD_DIR / "ud_adp_history.csv"
DRAFTERS_HISTORY = DASHBOARD_DIR / "drafters_adp_history.csv"
LATEST_SNAPSHOT  = DASHBOARD_DIR / "latest.json"
MANUAL_DIR       = REPO_ROOT / "_local" / "manual-snapshots"
MANUAL_DIR.mkdir(parents=True, exist_ok=True)

# Resolve drop file paths.
DROPS_DIR_ABS = (REPO_ROOT / DROPS_DIR).resolve() if not Path(DROPS_DIR).is_absolute() else Path(DROPS_DIR).resolve()
UD_DROP       = DROPS_DIR_ABS / UD_FILE
DK_DROP       = DROPS_DIR_ABS / DK_FILE
DRAFTERS_DROP = DROPS_DIR_ABS / DRAFTERS_FILE

# Sentinel floors (match pull_adp.py).
ADP_FLOORS = {"DK": 240.0, "UD": 216.0, "FFPC": float("inf"), "Drafters": float("inf")}

STACKED_HEADER = ["date", "name", "pos", "team", "adp", "source"]

# UD full team name -> 3-letter code (UD is the only source needing this mapping).
NFL_TEAM_CODE = {
    "Arizona Cardinals": "ARI", "Atlanta Falcons": "ATL", "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF", "Carolina Panthers": "CAR", "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN", "Cleveland Browns": "CLE", "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN", "Detroit Lions": "DET", "Green Bay Packers": "GB",
    "Houston Texans": "HOU", "Indianapolis Colts": "IND", "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC", "Las Vegas Raiders": "LV", "Los Angeles Chargers": "LAC",
    "Los Angeles Rams": "LAR", "Miami Dolphins": "MIA", "Minnesota Vikings": "MIN",
    "New England Patriots": "NE", "New Orleans Saints": "NO", "New York Giants": "NYG",
    "New York Jets": "NYJ", "Philadelphia Eagles": "PHI", "Pittsburgh Steelers": "PIT",
    "San Francisco 49ers": "SF", "Seattle Seahawks": "SEA", "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN", "Washington Commanders": "WAS",
}

# Verify inputs exist before doing anything destructive.
for label, p in [("UD", UD_DROP), ("DK", DK_DROP), ("Drafters", DRAFTERS_DROP)]:
    if not p.exists():
        raise FileNotFoundError(f"{label} drop not found: {p}")

print(f"Repo root:  {REPO_ROOT}")
print(f"Drops from: {DROPS_DIR_ABS}")
print(f"Today:      {TODAY}")
print("All three drop files present.")

## 3. Sanity check: DK is NFL, not MLB

This step exists because a DK MLB export slipped through once. NFL positions are QB/RB/WR/TE; MLB positions are P/IF/OF.

In [ ]:
with DK_DROP.open(encoding="utf-8") as f:
    positions = Counter()
    for r in csv.DictReader(f):
        if r.get("ADP", "").strip():
            positions[r.get("Position", "?")] += 1

print(f"DK position breakdown: {dict(positions)}")

nfl_positions = {"QB", "RB", "WR", "TE"}
mlb_positions = {"P", "IF", "OF"}
if set(positions.keys()) & mlb_positions:
    raise ValueError(f"DK file looks like MLB ({dict(positions)}). Aborting.")
if not (set(positions.keys()) & nfl_positions):
    raise ValueError(f"DK file has no NFL positions ({dict(positions)}). Aborting.")
print("NFL confirmed.")

## 4. Sync git remote

The morning cron already pushed today's stale auto rows. Pull first so we work from a clean base.

In [ ]:
def run_git(*args, check=True):
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        capture_output=True, text=True,
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{result.stderr}")
    return result.stdout.strip()

# Check if there are uncommitted changes that would block rebase.
status = run_git("status", "--porcelain")
if status:
    print("WARNING: uncommitted changes present. If pull fails, stash them first.")
    print(status)

run_git("pull", "--rebase", "--quiet")
print("Synced with origin/main.")

# Show what's currently in the history files for today (should be stale auto).
for src, path in [("dk", DK_HISTORY), ("ud", UD_HISTORY), ("drafters", DRAFTERS_HISTORY)]:
    with path.open(encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r["date"] == TODAY]
    by_src = Counter(r["source"] for r in rows)
    print(f"  {src} {TODAY}: {dict(by_src)}")

## 5. Move drops into `_local/manual-snapshots/`

Prefix with the compact date (`MMDD`) to avoid collisions with previous days.

In [ ]:
date_prefix = TODAY[5:7] + TODAY[8:10]  # e.g. '0718' for 2026-07-18

def move_with_prefix(src_path: Path) -> Path:
    dst = MANUAL_DIR / f"{date_prefix} {src_path.name}"
    if dst.exists():
        print(f"  already in _local: {dst.name} (skipping move)")
        return dst
    shutil.move(str(src_path), str(dst))
    print(f"  moved: {src_path.name} -> _local/manual-snapshots/{dst.name}")
    return dst

UD_DROP       = move_with_prefix(UD_DROP)
DK_DROP       = move_with_prefix(DK_DROP)
DRAFTERS_DROP = move_with_prefix(DRAFTERS_DROP)

## 6. Strip today's stale auto rows from DK/UD/Drafters history files

FFPC stays as stale auto (no manual FFPC file).

In [ ]:
for src, path in [("dk", DK_HISTORY), ("ud", UD_HISTORY), ("drafters", DRAFTERS_HISTORY)]:
    with path.open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    before = len(rows)
    keep = [r for r in rows if not (r["date"] == TODAY and r["source"] == "auto")]
    removed = before - len(keep)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=STACKED_HEADER, lineterminator="\n")
        w.writeheader()
        for r in keep:
            w.writerow(r)
    print(f"  {src}: removed {removed} stale auto rows for {TODAY}")

## 7. Parse each drop file and append manual rows

Each source has a slightly different schema:
- **UD**: `firstName + lastName`, `slotName`, `teamName` (full name), `adp` (blank = unranked)
- **DK**: `Name`, `Position`, `Team` (3-letter), `ADP` (7-decimal, round to 1)
- **Drafters**: `name`, `position`, `team abbr`, `ADP` (`0` = sentinel for undrafted)

In [ ]:
def _append_history(path: Path, rows: list[list[str]]) -> None:
    with path.open("a", encoding="utf-8", newline="") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerows(rows)

def parse_ud(path: Path) -> list[list[str]]:
    out = []
    with path.open(encoding="utf-8", newline="") as f:
        for r in csv.DictReader(f):
            try:
                adp = float(r["adp"])
            except (TypeError, ValueError, KeyError):
                continue
            if adp >= ADP_FLOORS["UD"]:
                continue
            name = (r["firstName"].strip() + " " + r["lastName"].strip()).strip()
            pos  = (r.get("slotName") or "").strip()
            team_full = (r.get("teamName") or "").strip()
            team = NFL_TEAM_CODE.get(team_full, "")
            if not name:
                continue
            out.append([TODAY, name, pos, team, f"{adp:.1f}", "manual"])
    return out

def parse_dk(path: Path) -> list[list[str]]:
    out = []
    with path.open(encoding="utf-8", newline="") as f:
        for r in csv.DictReader(f):
            try:
                adp = float(r["ADP"])
            except (TypeError, ValueError, KeyError):
                continue
            if adp >= ADP_FLOORS["DK"]:
                continue
            name = (r.get("Name") or "").strip()
            pos  = (r.get("Position") or "").strip()
            team = (r.get("Team") or "").strip()
            if not name:
                continue
            out.append([TODAY, name, pos, team, f"{adp:.1f}", "manual"])
    return out

def parse_drafters(path: Path) -> list[list[str]]:
    out = []
    with path.open(encoding="utf-8", newline="") as f:
        for r in csv.DictReader(f):
            try:
                adp = float(r["ADP"])
            except (TypeError, ValueError, KeyError):
                continue
            # Drafters encodes 'undrafted' as ADP=0, not blank. Filter <= 0.
            if adp <= 0:
                continue
            if adp >= ADP_FLOORS["Drafters"]:
                continue
            name = (r.get("name") or "").strip()
            pos  = (r.get("position") or "").strip()
            team = (r.get("team abbr") or "").strip()
            if not name:
                continue
            out.append([TODAY, name, pos, team, f"{adp:.1f}", "manual"])
    return out

for label, path, parsed_rows in [
    ("DK",       DK_HISTORY,       parse_dk(DK_DROP)),
    ("UD",       UD_HISTORY,       parse_ud(UD_DROP)),
    ("Drafters", DRAFTERS_HISTORY, parse_drafters(DRAFTERS_DROP)),
]:
    if not parsed_rows:
        raise ValueError(f"{label}: 0 parsed rows. Schema mismatch or empty file.")
    _append_history(path, parsed_rows)
    print(f"  {label}: appended {len(parsed_rows)} manual rows -> {path.name}")

## 8. Rebuild `latest.json`

Merge today's DK/UD/Drafters manual rows across sources. Carry FFPC values forward from the current `latest.json` since no fresh FFPC data was provided.

In [ ]:
source_paths = {
    "DK":       DK_HISTORY,
    "UD":       UD_HISTORY,
    "Drafters": DRAFTERS_HISTORY,
}
by_src_today: dict[str, dict[str, dict]] = {}
for src, path in source_paths.items():
    with path.open(encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    manual = [r for r in rows if r["date"] == TODAY and r["source"] == "manual"]
    auto   = [r for r in rows if r["date"] == TODAY and r["source"] == "auto"]
    chosen = manual or auto
    by_src_today[src] = {r["name"].strip(): r for r in chosen}

# Carry FFPC forward from the current latest.json.
ffpc_carryover: dict[str, float] = {}
if LATEST_SNAPSHOT.exists():
    old = json.loads(LATEST_SNAPSHOT.read_text(encoding="utf-8"))
    for p in old.get("players", []):
        v = p.get("adps", {}).get("FFPC")
        if isinstance(v, (int, float)):
            ffpc_carryover[p["name"]] = float(v)

by_name: dict[str, dict] = {}
for src in ("DK", "UD", "Drafters"):
    for name, row in by_src_today[src].items():
        entry = by_name.setdefault(name, {
            "name": name,
            "pos":  row["pos"],
            "team": row["team"],
            "adps": {},
        })
        try:
            val = float(row["adp"])
        except ValueError:
            continue
        if val <= 0:
            continue
        if val >= ADP_FLOORS.get(src, float("inf")):
            continue
        entry["adps"][src] = val
        if not entry["pos"]  and row.get("pos"):  entry["pos"]  = row["pos"]
        if not entry["team"] and row.get("team"): entry["team"] = row["team"]

for name, ffpc_adp in ffpc_carryover.items():
    if ffpc_adp >= ADP_FLOORS["FFPC"]:
        continue
    if name in by_name:
        by_name[name]["adps"]["FFPC"] = ffpc_adp

players = [p for p in by_name.values() if p["adps"]]
players.sort(key=lambda p: min(p["adps"].values()))

snapshot = {
    "pulled_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "date":      TODAY,
    "players":   players,
}
LATEST_SNAPSHOT.write_text(
    json.dumps(snapshot, separators=(",", ":")) + "\n",
    encoding="utf-8",
)
print(f"Wrote {LATEST_SNAPSHOT.name}: {len(players)} players, date={TODAY}")

## 9. Verify

Sanity-check counts, source coverage, and top 3 players before committing.

In [ ]:
d = json.loads(LATEST_SNAPSHOT.read_text(encoding="utf-8"))
print(f"latest.json: date={d['date']}  players={len(d['players'])}")

sources = Counter()
for p in d["players"]:
    for s in p["adps"]:
        sources[s] += 1
print(f"source coverage: {dict(sources)}")

print("top 3:")
for p in d["players"][:3]:
    print(f"  {p['name']} ({p['pos']} {p['team']}): {p['adps']}")

print()
for src, path in [("dk", DK_HISTORY), ("ud", UD_HISTORY),
                  ("ffpc", DASHBOARD_DIR / "ffpc_adp_history.csv"),
                  ("drafters", DRAFTERS_HISTORY)]:
    with path.open(encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r["date"] == TODAY]
    print(f"  {src} {TODAY}: {dict(Counter(r['source'] for r in rows))}")

# Simple sanity gate: expect top player to be a familiar name.
top_name = d["players"][0]["name"] if d["players"] else ""
familiar = {"Bijan Robinson", "Jahmyr Gibbs", "Ja'Marr Chase", "Puka Nacua", "Justin Jefferson"}
if top_name not in familiar:
    print()
    print(f"WARNING: top player is {top_name!r} — not one of the usual top-tier RBs/WRs.")
    print("         Inspect before committing.")

## 10. Commit and push

Uses targeted `git add` (never `-A`) so nothing else in the repo root gets swept up.

In [ ]:
paths_to_add = [
    "dashboards/best-ball-prices/dk_adp_history.csv",
    "dashboards/best-ball-prices/ud_adp_history.csv",
    "dashboards/best-ball-prices/drafters_adp_history.csv",
    "dashboards/best-ball-prices/latest.json",
]
run_git("add", *paths_to_add)

counts_lines = []
for label, src in [("DK", "DK"), ("UD", "UD"), ("Drafters", "Drafters")]:
    counts_lines.append(f"  {label + ':':<10} {sources[src]} manual")
counts_lines.append(f"  {'FFPC:':<10} {sources['FFPC']} stale auto (unchanged)")

commit_msg = (
    f"Manual {TODAY} ADP snapshot for DK, UD, Drafters (via notebook)\n"
    f"\n"
    f"Ran scripts/manual_update.ipynb. DK NFL position check passed\n"
    f"before running.\n"
    f"\n"
    f"Counts:\n"
    + "\n".join(counts_lines) + "\n"
    f"\n"
    f"latest.json: {len(d['players'])} players. Top player: {top_name}.\n"
)

commit_result = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "commit", "-m", commit_msg],
    capture_output=True, text=True,
)
print(commit_result.stdout)
if commit_result.returncode != 0:
    print(commit_result.stderr, file=sys.stderr)
    raise RuntimeError("git commit failed")

push_result = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "push"],
    capture_output=True, text=True,
)
print(push_result.stdout)
if push_result.returncode != 0:
    print(push_result.stderr, file=sys.stderr)
    raise RuntimeError("git push failed")

print(f"\nDone. {TODAY} manual snapshot is live in ~30-60s.")